# 03 — Grid run (driver over `scripts/run_synth_grid.py`)

Replaces v2's `02_synthetic_baselines` (archived in `notebooks/archive/v2/`).

The grid is split into **units**: (model, naming, task, strategy, label mode, k, seed).
- **One demonstration set per unit** for a query-agnostic strategy. It is reused for every query of the task in every environment.
- **One set per query** for feature kNN (`similarity`).
- **Paired comparisons:** demonstration sets depend only on (task, strategy, seed, k), so comparisons across models, namings and label modes are paired.
- **Prefix caching:** each set is one prompt prefix, and the queries plus the content-free calibration query are scored against its cache.
- **Resumable:** rows are appended after each unit, so rerunning a command resumes where it stopped.

Design: `docs/generator_spec.pdf`, Parts "Demonstration selection", "Model inference" and "Experiment orchestration".

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.utils.config import load_config, resolve_path  # first src import: sets the thread and MPS settings

config = load_config()

import subprocess

import pandas as pd

from src.data.suites import load_suite
from src.experiments.synth_grid import STRATEGIES, enumerate_units
from src.utils.results_schema import RESULTS_COLUMNS_V3

RESULTS = resolve_path(config.paths.results)

## 1. Units

In [2]:
eval_ids = [t["task_id"] for t in load_suite("eval", config)["tasks"]]
QI = "qwen2.5-7b-instruct"
NAMINGS = ["abstract", "aligned", "flipped"]
CORE = ["zero_shot", "random", "label_diversity", "feature_range", "rule_diversity", "counter_spurious", "counter_prior"]
plan = {
    "core strategies (3 seeds)": enumerate_units([QI], NAMINGS, eval_ids, CORE, ["gold"], [config.selection.k], 3),
    "random-label control": enumerate_units([QI], NAMINGS, eval_ids, ["label_diversity"], ["shuffled"], [config.selection.k], 3),
    "similarity (1 seed)": enumerate_units([QI], NAMINGS, eval_ids, ["similarity"], ["gold"], [config.selection.k], 1),
}
for name, units in plan.items():
    print(f"{name}: {len(units)} units")
print(f"Qwen2.5-7B-Instruct P4 grid: {sum(len(u) for u in plan.values())} units "
      f"({len(eval_ids)} tasks x {len(NAMINGS)} namings, k = {config.selection.k})")
for u in plan["core strategies (3 seeds)"][:3]:
    print("  ", u.key)

core strategies (3 seeds): 1368 units
random-label control: 216 units
similarity (1 seed): 72 units
Qwen2.5-7B-Instruct P4 grid: 1656 units (24 tasks x 3 namings, k = 8)
   qwen2.5-7b-instruct|abstract|eval_0000|zero_shot|gold|k0|s0
   qwen2.5-7b-instruct|abstract|eval_0000|random|gold|k8|s0
   qwen2.5-7b-instruct|abstract|eval_0000|random|gold|k8|s1


## 2. Smoke run (P1)

The run covers 2 tasks, 4 queries per environment (2 per class) and 2 strategies on Qwen2.5-7B-Instruct.
The script skips units that are already saved, and it loads a model only if that model still has units to run.

In [3]:
cmd = [sys.executable, "../scripts/run_synth_grid.py", "--run", "smoke", "--models", "qwen2.5-7b-instruct",
       "--tasks", "2", "--queries", "4", "--strategies", "random,counter_spurious"]
out = subprocess.run(cmd, capture_output=True, text=True)
print(out.stdout[-1500:] or out.stderr[-1500:])

12 units -> /Users/chenuka/Documents/USYD/LLM-ICL-OOD-Honours/sata-project/results/v3/synthetic/smoke.parquet
qwen2.5-7b-instruct: all units already saved



## 3. Check the results

The checks cover the schema, one set of rows per unit, and the resume test.
`smoke_resume` was killed after 6 of its 26 units and then rerun, so it should hold each unit exactly once.

In [4]:
for name in ["smoke", "smoke_resume"]:
    df = pd.read_parquet(RESULTS / f"{name}.parquet")
    dup = int(df.duplicated(["unit_key", "env", "query_id"]).sum())
    print(f"{name}: {len(df)} rows, {df.unit_key.nunique()} units, schema ok: {list(df.columns) == RESULTS_COLUMNS_V3}, "
          f"duplicate rows: {dup}, device: {df.device_name.iloc[0]} ({df.dtype.iloc[0]})")

smoke: 144 rows, 12 units, schema ok: True, duplicate rows: 0, device: Apple M3 Max (MPS) (bfloat16)
smoke_resume: 312 rows, 26 units, schema ok: True, duplicate rows: 0, device: Apple M3 Max (MPS) (bfloat16)


In [5]:
df = pd.read_parquet(RESULTS / "smoke_resume.parquet")
summary = (df.assign(correct=df.prediction == df.label, correct_cal=df.prediction_cal == df.label)
             .groupby(["strategy", "env"])[["correct", "correct_cal", "p1"]].mean().round(2))
print("Only 2 tasks with 4 queries per environment: a pipeline check, not a result.")
summary

Only 2 tasks with 4 queries per environment: a pipeline check, not a result.


correct  correct_cal    p1
strategy         env                                          
counter_spurious covariate             0.46         0.50  0.95
                 id                    0.50         0.58  0.96
                 spurious_reversal     0.46         0.33  0.87
label_diversity  covariate             0.46         0.54  0.95
                 id                    0.50         0.62  0.99
                 spurious_reversal     0.46         0.33  0.86
random           covariate             0.50         0.58  0.95
                 id                    0.50         0.50  0.95
                 spurious_reversal     0.50         0.38  0.95
similarity       covariate             0.54         0.54  0.95
                 id                    0.58         0.62  0.86
                 spurious_reversal     0.50         0.38  0.94
zero_shot        covariate             0.50         0.00  0.52
                 id                    0.62         0.00  0.60
                 spurious_reversal     0.62         0.00  0.56

## 4. Full grid (P4)

P4 runs from a terminal as resumable stages (`scripts/run_p4.sh`; the stage list is in the script and in `docs/research_plan.md`):

```
caffeinate -dimsu sh scripts/run_p4.sh abstract named_core named_priors prior rq3 named_extra base rq3_more
```

Everything goes to `results/v3/synthetic/p4/`: `grid.parquet` (Qwen2.5-7B-Instruct), `grid_base.parquet` (Qwen2.5-7B base), `rq3.parquet` with `rq3_rankings.parquet`, and `logs/`.
`similarity` builds a separate prompt for every query, so it has no prefix to share; it costs about 6× as much per unit.

The cell below reports progress against the plan in Section 1.

In [6]:
path = RESULTS / "p4" / "grid.parquet"
if path.exists():
    done = set(pd.read_parquet(path, columns=["unit_key"])["unit_key"].unique())
    for name, units in plan.items():
        by = pd.DataFrame([{"naming": u.naming, "strategy": u.strategy, "done": u.key in done} for u in units])
        print(f"{name}: {by.done.sum()}/{len(by)} units")
        print(by.pivot_table(index="strategy", columns="naming", values="done", aggfunc="sum").to_string(), "\n")
    g = pd.read_parquet(path, columns=["unit_key", "naming", "strategy", "unit_seconds", "n_prompt_tokens"])
    u = g.drop_duplicates("unit_key")
    display(u.groupby(["naming", "strategy"])["unit_seconds"].mean().unstack("naming").round(1))
else:
    print("P4 has not started")
print(open(RESULTS / "p4" / "logs" / "steps.log").read() if (RESULTS / "p4" / "logs" / "steps.log").exists() else "")

core strategies (3 seeds): 58/1368 units
naming            abstract  aligned  flipped
strategy                                    
counter_prior            0        0        0
counter_spurious         9        0        0
feature_range           10        0        0
label_diversity         10        0        0
random                  10        0        0
rule_diversity           9        0        0
zero_shot               10        0        0 

random-label control: 0/216 units
naming           abstract  aligned  flipped
strategy                                   
label_diversity         0        0        0 

similarity (1 seed): 0/72 units
naming      abstract  aligned  flipped
strategy                              
similarity         0        0        0 



naming,abstract
strategy,
counter_spurious,24.0
feature_range,24.8
label_diversity,25.7
random,25.9
rule_diversity,24.0
zero_shot,21.4


2026-09-30 15:30 grid abstract: core strategies

